In [ ]:
#Just sorting our dataset images into their labels and into training, validation and testing sets
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
source_folder = DATA_BASE_PATH          # folder with all your images
sorted_folder = "/home/beylessen/Desktop/PFA2/Sorted"     # output folder
splits = ["train", "val", "test"]      # dataset splits

# Step 1: Sort files into folders by label (prefix before '_')
all_files = os.listdir(source_folder)
labels = set()

for f in all_files:
    if os.path.isfile(os.path.join(source_folder, f)):
        if "_" in f:
            label = f.split("_")[0]  # take everything before first underscore
        else:
            label = "unknown"
        labels.add(label)
        label_folder = os.path.join(sorted_folder, label)
        os.makedirs(label_folder, exist_ok=True)
        shutil.copy(os.path.join(source_folder, f), os.path.join(label_folder, f))

print(f"Files sorted into label folders: {labels}")

# Step 2: Split each label folder into train/val/test
for label in labels:
    label_path = os.path.join(sorted_folder, label)
    files = os.listdir(label_path)
    
    if len(files) == 0:
        continue  # skip empty folders
    
    # Split: 70% train, 20% val, 10% test
    train_files, temp_files = train_test_split(files, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=1/3, random_state=42)
    
    for split_name, split_files in zip(splits, [train_files, val_files, test_files]):
        split_folder = os.path.join(sorted_folder, split_name, label)
        os.makedirs(split_folder, exist_ok=True)
        for f in split_files:
            shutil.move(os.path.join(label_path, f), os.path.join(split_folder, f))

print("Dataset sorted by label and split into train/val/test!")
#Training (70%) Validation (20%) Testing (10%)

In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm  # nice progress bar

# Path to your TRAIN images only
TRAIN_DIR = "/home/beylessen/Desktop/PFA2/Sorted/train"

# Lists to hold per-image means/stds
means = []
stds = []

# Iterate over each class folder
for cls in os.listdir(TRAIN_DIR):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    
    for fname in tqdm(os.listdir(cls_path)):
        img_path = os.path.join(cls_path, fname)
        img = Image.open(img_path).convert("RGB")  # make sure image is RGB
        img = np.array(img) / 255.0               # scale pixels to 0-1
        
        # Compute per-channel mean and std for this image
        means.append(img.mean(axis=(0,1)))   # axis=(0,1) collapses height & width
        stds.append(img.std(axis=(0,1)))

# Convert to numpy array and average across all images
mean = np.mean(means, axis=0)
std  = np.mean(stds, axis=0)

print("Custom dataset mean (R,G,B):", mean)
print("Custom dataset std  (R,G,B):", std)
